# Train group R‑GCN V3 baseline: HC3 → GPT‑3.5 Mixed (0% noise), batch 64

This notebook trains only the recognized `baseline` V3 (`QuangV3` with Structural Features and Syntax‑Aware Swap), not CMixStyle, CorrectedStruct, R‑Drop, MeanLogit, or another ablation. It selects the checkpoint using HC3 validation only, then evaluates only the clean GPT‑3.5 Mixed split.

`--batch-size 64` is the effective optimizer batch. `--micro-batch-size 8` is retained solely to fit the graph donor pool in GPU memory.

In [ ]:
# Kaggle Python 3.13 needs current binary wheels. Do not pin transformers==4.44.2.
!pip install -q --upgrade torch-geometric transformers spacy scikit-learn tqdm
!python -m spacy download en_core_web_sm

import torch
import torch_geometric
import transformers
import spacy
print({
    "torch": torch.__version__,
    "torch_geometric": torch_geometric.__version__,
    "transformers": transformers.__version__,
    "spacy": spacy.__version__,
})


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

WORK = Path("/kaggle/working/group_v3_hc3_to_gpt35_bs64")
REPO = WORK / "PRDetect-Research"
BRANCH = "feat/rgcn-rdrop"
SEED = 2026
BATCH_SIZE = 64
MICRO_BATCH_SIZE = 8
EPOCHS = 10
TEST_SPLIT = "gpt3.5_mixed_test_split"
MODEL_DIR = WORK / "model"
EVAL_DIR = WORK / "evaluation"
if REPO.exists():
    shutil.rmtree(REPO)
WORK.mkdir(parents=True, exist_ok=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                "https://github.com/tphan1420/PRDetect-Research.git", str(REPO)], check=True)
MODEL_DIR.mkdir(exist_ok=True)
EVAL_DIR.mkdir(exist_ok=True)
assert (REPO / "train_v3.py").is_file() and (REPO / "evaluate_v3_quang.py").is_file()
assert (REPO / "data/original_text/benchmarks/gpt35_mixed/gpt3.5_mixed_test_split.json").is_file()
print({"commit": subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip(),
       "variant": "baseline", "seed": SEED, "effective_batch_size": BATCH_SIZE,
       "micro_batch_size": MICRO_BATCH_SIZE, "test": TEST_SPLIT})


In [ ]:
# The test graph is deliberately not built until after V3 training.
subprocess.run([sys.executable, "building_graph.py", "--file", "hc3_train", "hc3_val"], cwd=REPO, check=True)


In [ ]:
subprocess.run(
    [sys.executable, "train_v3.py", "--variants", "baseline", "--seeds", str(SEED),
     "--graph-dir", str(REPO / "graph_data"), "--output-dir", str(MODEL_DIR),
     "--train-file", "hc3_train", "--val-file", "hc3_val", "--struct-mode", "low",
     "--batch-size", str(BATCH_SIZE), "--micro-batch-size", str(MICRO_BATCH_SIZE),
     "--val-batch-size", "64", "--epochs", str(EPOCHS), "--patience", "10", "--device", "cuda"],
    cwd=REPO,
    check=True,
)
CHECKPOINT = MODEL_DIR / f"v3_baseline_seed{SEED}.pt"
TRAIN_METADATA = CHECKPOINT.with_suffix(".train.json")
assert CHECKPOINT.is_file() and TRAIN_METADATA.is_file()


In [ ]:
subprocess.run([sys.executable, "building_graph.py", "--file", TEST_SPLIT], cwd=REPO, check=True)
GRAPH = REPO / "graph_data" / f"{TEST_SPLIT}.pkl"
assert GRAPH.is_file()

# Use the repository evaluator's strict loader and its internal one-split worker.
from types import SimpleNamespace
import importlib.util
spec = importlib.util.spec_from_file_location("v3_eval", REPO / "evaluate_v3_quang.py")
v3_eval = importlib.util.module_from_spec(spec)
spec.loader.exec_module(v3_eval)
args = SimpleNamespace(variant="baseline", train_domain="hc3", checkpoint_dir=str(MODEL_DIR), seeds=[SEED],
                       batch_size=BATCH_SIZE, struct_mode="low", device="cuda", graph_budget_root=str(WORK))
config = v3_eval.make_config(args)
CONFIG = EVAL_DIR / "config.json"
v3_eval.write_json(CONFIG, config)
expected_samples = sum(1 for line in (REPO / "data/original_text/benchmarks/gpt35_mixed/gpt3.5_mixed_test_split.json").open(encoding="utf-8") if line.strip())
assert expected_samples == 1000
REPORT = EVAL_DIR / "gpt35_0pct_report.json"
subprocess.run([sys.executable, "evaluate_v3_quang.py", "_test", "--config", str(CONFIG), "--seed", str(SEED),
                "--graph", str(GRAPH), "--expected-samples", str(expected_samples),
                "--signature", "single_gpt35_0pct_after_hc3_training", "--report", str(REPORT)], cwd=REPO, check=True)
assert REPORT.is_file()


In [ ]:
import zipfile
report = json.loads(REPORT.read_text(encoding="utf-8"))
archive = WORK / "v3_baseline_hc3_to_gpt35_0pct_batch64_results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as out:
    for path in (CHECKPOINT, TRAIN_METADATA, CONFIG, REPORT):
        out.write(path, path.name)
print(json.dumps({key: report[key] for key in ("samples", "accuracy", "auc", "f1", "recall_ai", "recall_human", "threshold", "validation_threshold")}, indent=2))
print("ZIP:", archive)
